# Analyse RCEMIP — Simulation `large300`
## Transport vertical de la quantité de mouvement par la turbulence convective

**Objectifs :**
1. Chargement robuste (lazy, chunked) des fichiers 3D volumieux
2. Détection de l'auto-agrégation convective via la PRW
3. Classification sec / humide et cartes 2D XY
4. Bilan de quantité de mouvement global et conditionnel (par régime turbulent)
5. Densité ρ₀ calculée à partir de la pression et de la température virtuelle


## 1. Librairies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import xarray as xr
import dask
import os

# Paramètres globaux matplotlib
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

# Constantes physiques
Rd      = 287.05   # J kg⁻¹ K⁻¹  — constante gaz sec
Rv      = 461.5    # J kg⁻¹ K⁻¹  — constante vapeur d'eau
EPSILON = Rd / Rv  # ≈ 0.622 (ratio des masses molaires)

print('Librairies chargées.')
print(f'  Rd = {Rd} J/kg/K,  Rv = {Rv} J/kg/K,  ε = Rd/Rv = {EPSILON:.4f}')


## 2. Chargement des données 3D (lazy + chunked)

La grille large est typiquement **300×300×74×100** points (x, y, z, t).  
Charger tout en RAM d'un coup provoque un crash mémoire.  
Solution : `xr.open_dataset(..., chunks=CHUNKS)` — les données restent sur disque  
et ne sont calculées que quand on les demande explicitement (`.compute()` ou `.load()`).  
On travaille **toujours** sur des tranches temporelles ou spatiales limitées.


In [ ]:
# Dossier contenant les fichiers NetCDF 3D
DIR_3D = '3D'
DIR_1D = '1D'

# Taille des chunks : adapter à la RAM disponible.
# Ici on découpe le temps en blocs de 10 pas → un bloc ≈ quelques centaines de Mo.
CHUNKS = {'time': 10}

def open_lazy(varname):
    """Ouvre un fichier NetCDF en mode paresseux (lazy) avec chunks Dask."""
    path = os.path.join(DIR_3D, f'MESONH_RCE_large300_3D_{varname}.nc')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Fichier introuvable : {path}')
    return xr.open_dataset(path, chunks=CHUNKS)

# --- Variables 3D (chargement lazy, pas encore en RAM) ---
ds_ua   = open_lazy('ua')    # vent zonal u  [m/s]
ds_va   = open_lazy('va')    # vent méridien v  [m/s]
ds_wa   = open_lazy('wa')    # vitesse verticale w  [m/s]
ds_ta   = open_lazy('ta')    # température T  [K]
ds_pa   = open_lazy('pa')    # pression p  [Pa]
ds_hus  = open_lazy('hus')   # humidité spécifique qv  [kg/kg]
ds_clw  = open_lazy('clw')   # eau liquide nuageuse  [kg/kg]
ds_cli  = open_lazy('cli')   # glace nuageuse  [kg/kg]
ds_plw  = open_lazy('plw')   # eau liquide précipitante  [kg/kg]
ds_pli  = open_lazy('pli')   # glace précipitante  [kg/kg]
ds_hur  = open_lazy('hur')   # humidité relative  [%]

# --- Extraction des DataArrays ---
u   = ds_ua['ua']
v   = ds_va['va']
w   = ds_wa['wa']
T   = ds_ta['ta']
p   = ds_pa['pa']
qv  = ds_hus['hus']
clw = ds_clw['clw']
cli = ds_cli['cli']
plw = ds_plw['plw']
pli = ds_pli['pli']

# --- Axes de coordonnées ---
# On suppose que les dimensions sont (time, altitude, y, x)
# On récupère leurs noms de façon robuste
dim_t   = u.dims[0]   # 'time'
dim_z   = u.dims[1]   # 'altitude'
dim_y   = u.dims[2]   # 'y'
dim_x   = u.dims[3]   # 'x'
alt     = u[dim_z].values  # vecteur altitude [m]

print('Données ouvertes en mode lazy (aucune RAM consommée pour l instant).')
print(f'  Dimensions : {dict(u.sizes)}')
print(f'  Axes       : t={dim_t}, z={dim_z}, y={dim_y}, x={dim_x}')


## 3. Eau précipitable (PRW) et auto-agrégation convective

La **PRW** (Precipitable Water, ou colonne d'eau intégrée) est le premier  
indicateur d'auto-agrégation : dans une simulation agrégée, la distribution de PRW  
devient bimodale — des colonnes très sèches coexistent avec des colonnes très humides.

**Calcul :** PRW = ∫ ρ₀ qv dz  
avec ρ₀ = p / (Rd · Tv) et Tv = T · (1 + qv/ε) / (1 + qv)


In [ ]:
# ======================================================================
# 3.1  Calcul de ρ₀(z) à partir de la température virtuelle
# ======================================================================
# On utilise le dernier tiers de la simulation (état stationnaire)
# On charge un seul pas de temps à la fois pour limiter la mémoire

n_times = u.sizes[dim_t]
idx_stat = slice(int(2 * n_times / 3), None)  # dernier tiers

# Température virtuelle : Tv = T * (1 + qv/ε) / (1 + qv)  ≈  T * (1 + (1/ε - 1)*qv)
# On utilise la forme exacte
T_stat  = T.isel({dim_t: idx_stat})
p_stat  = p.isel({dim_t: idx_stat})
qv_stat = qv.isel({dim_t: idx_stat})

Tv_stat = T_stat * (1.0 + qv_stat / EPSILON) / (1.0 + qv_stat)
rho_stat = p_stat / (Rd * Tv_stat)  # [kg/m³]

# Moyenne temporelle de ρ₀ → profil vertical de référence
rho0 = rho_stat.mean(dim=[dim_t, dim_y, dim_x]).compute()

print('Profil ρ₀ calculé.')
print(f'  ρ₀ surface ≈ {float(rho0.isel({dim_z: 0})):.3f} kg/m³')
print(f'  ρ₀ à 10 km  ≈ {float(rho0.sel({dim_z: 10000}, method="nearest")):.3f} kg/m³')

fig, ax = plt.subplots(figsize=(4, 6))
ax.plot(rho0.values, alt, color='steelblue', lw=2)
ax.set_xlabel('ρ₀ (kg/m³)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Profil vertical de la densité ρ₀')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ======================================================================
# 3.2  Calcul de la PRW pour chaque colonne et chaque pas de temps
# ======================================================================
# PRW(x, y, t) = Σ_z  ρ₀(z) * qv(x, y, z, t) * Δz
# On utilise la méthode trapézoïdale via xarray.integrate sur z

# On travaille sur l'état stationnaire uniquement
qv_s = qv.isel({dim_t: idx_stat})  # lazy, pas encore chargé

# ρ₀ doit avoir la même dimension z que qv → on l'aligne
rho0_da = rho0  # DataArray 1D en altitude

# Intégrale verticale : PRW = ∫ ρ₀ · qv · dz
prw = (rho0_da * qv_s).integrate(coord=dim_z)  # [kg/m²]

# Moyenne temporelle de PRW
prw_mean = prw.mean(dim=dim_t).compute()

print(f'PRW calculée. Plage : {float(prw_mean.min()):.1f} — {float(prw_mean.max()):.1f} kg/m²')


In [ ]:
# ======================================================================
# 3.3  Carte 2D de la PRW moyenne (signe d auto-agrégation ?)
# ======================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Carte 2D
ax = axes[0]
im = ax.pcolormesh(prw_mean.values, cmap='Blues', vmin=0)
cb = fig.colorbar(im, ax=ax, label='PRW (kg/m²)')
ax.set_title('Carte de la PRW moyenne (état stationnaire)', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')

# Distribution (histogramme) — signal d auto-agrégation = distribution bimodale
ax2 = axes[1]
prw_flat = prw_mean.values.ravel()
ax2.hist(prw_flat, bins=60, color='steelblue', edgecolor='white', lw=0.4)
ax2.axvline(np.median(prw_flat), color='red', lw=1.5, linestyle='--', label=f'Médiane = {np.median(prw_flat):.1f}')
ax2.set_xlabel('PRW (kg/m²)')
ax2.set_ylabel('Nombre de colonnes')
ax2.set_title('Distribution de la PRW\n(bimodale = auto-agrégation)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Définition des régions sèches et humides

On calibre visuellement le seuil sur la distribution de PRW.  
Typiquement dans les simulations RCE agrégées :  
- **Humide** : PRW > PRW_seuil (colonnes convectives)  
- **Sec** : PRW < PRW_seuil (colonnes subsidentes)


In [ ]:
# ======================================================================
# 4.1  Choix du seuil (à ajuster selon la distribution observée ci-dessus)
# ======================================================================
# Valeur initiale : médiane de la distribution
PRW_SEUIL = float(np.median(prw_mean.values))
print(f'Seuil PRW retenu : {PRW_SEUIL:.1f} kg/m²')
print('Modifier PRW_SEUIL si nécessaire après visualisation de la distribution.')

# Masques 2D (x, y) — True là où la colonne est humide / sèche
masque_humide = prw_mean > PRW_SEUIL
masque_sec    = prw_mean <= PRW_SEUIL

frac_humide = float(masque_humide.mean()) * 100
frac_sec    = float(masque_sec.mean())    * 100
print(f'Fraction humide : {frac_humide:.1f}%  |  Fraction sèche : {frac_sec:.1f}%')


In [ ]:
# ======================================================================
# 4.2  Carte sec / humide
# ======================================================================
fig, ax = plt.subplots(figsize=(7, 6))

cmap_bw = mcolors.ListedColormap(['#d9e8f5', '#08306b'])  # sec = bleu clair, humide = bleu foncé
im = ax.pcolormesh(masque_humide.values.astype(int), cmap=cmap_bw, vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax, ticks=[0.25, 0.75])
cbar.ax.set_yticklabels(['Sec', 'Humide'])

ax.set_title(f'Carte sec / humide  (seuil PRW = {PRW_SEUIL:.1f} kg/m²)', fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## 5. Profil vertical du vent horizontal moyen

In [ ]:
# ======================================================================
# 5.1  État initial (premiers 5 pas de temps)
# ======================================================================
u_init = u.isel({dim_t: slice(0, 5)}).mean(dim=[dim_t, dim_y, dim_x]).compute()
v_init = v.isel({dim_t: slice(0, 5)}).mean(dim=[dim_t, dim_y, dim_x]).compute()

fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(u_init.values, alt, color='royalblue', lw=2, label=r'$\overline{u}$')
ax.plot(v_init.values, alt, color='seagreen', lw=2, label=r'$\overline{v}$')
ax.plot(np.sqrt(u_init**2 + v_init**2).values, alt, 'k--', lw=1.5, label='Module')
ax.axvline(0, color='red', alpha=0.4, lw=1)
ax.set_xlabel('Vitesse (m/s)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Profil du vent horizontal — état initial', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ======================================================================
# 5.2  État stationnaire (dernier tiers de la simulation)
# ======================================================================
u_stat = u.isel({dim_t: idx_stat}).mean(dim=[dim_t, dim_y, dim_x]).compute()
v_stat = v.isel({dim_t: idx_stat}).mean(dim=[dim_t, dim_y, dim_x]).compute()

fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(u_stat.values, alt, color='royalblue', lw=2, label=r'$\overline{u}$')
ax.plot(v_stat.values, alt, color='seagreen', lw=2, label=r'$\overline{v}$')
ax.plot(np.sqrt(u_stat**2 + v_stat**2).values, alt, 'k--', lw=1.5, label='Module')
ax.axvline(0, color='red', alpha=0.4, lw=1)
ax.set_xlabel('Vitesse (m/s)')
ax.set_ylabel('Altitude (m)')
ax.set_title('Profil du vent horizontal — état stationnaire', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 6. Bilan de quantité de mouvement — global

L'équation de quantité de mouvement zonale moyennée horizontalement s'écrit :

$$
\frac{\partial \overline{u}}{\partial t} = - \frac{1}{\rho_0}\frac{\partial}{\partial z}(\rho_0 \, \overline{u'w'}) + \text{adv} + \text{pression} + \text{rappel}
$$

Ici on calcule le **tenseur de Reynolds** ρ₀⟨u'w'⟩ et sa divergence verticale,  
qui représente la force nette exercée sur le vent moyen par la turbulence.


In [ ]:
# ======================================================================
# 6.1  Calcul du flux de Reynolds ρ₀ <u'w'> en traitant temps par temps
# ======================================================================
# Pour éviter de charger toutes les variables simultanément,
# on boucle sur des blocs temporels et on accumule la moyenne.

BLOC = 10  # nombre de pas de temps par bloc (réduire si RAM insuffisante)

flux_uw_sum = np.zeros(len(alt))  # accumulateur
flux_vw_sum = np.zeros(len(alt))
n_processed = 0

t_start = int(2 * n_times / 3)  # on part de l état stationnaire

for t0 in range(t_start, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    # Chargement du bloc en RAM
    u_blk = u.isel({dim_t: sl}).load()  # (bloc, z, y, x)
    v_blk = v.isel({dim_t: sl}).load()
    w_blk = w.isel({dim_t: sl}).load()

    # Moyenne horizontale → profil (bloc, z)
    u_moy_blk = u_blk.mean(dim=[dim_y, dim_x])
    v_moy_blk = v_blk.mean(dim=[dim_y, dim_x])
    w_moy_blk = w_blk.mean(dim=[dim_y, dim_x])

    # Anomalies turbulentes : φ' = φ - <φ>_xy
    u_prime = u_blk - u_moy_blk
    v_prime = v_blk - v_moy_blk
    w_prime = w_blk - w_moy_blk

    # Flux <u'w'> et <v'w'> — moyenne horizontale puis pondération par ρ₀
    flux_uw_blk = (rho0 * u_prime * w_prime).mean(dim=[dim_y, dim_x])
    flux_vw_blk = (rho0 * v_prime * w_prime).mean(dim=[dim_y, dim_x])

    # Accumulation (moyenne temporelle en cours)
    n_bloc = t1 - t0
    flux_uw_sum += flux_uw_blk.mean(dim=dim_t).values * n_bloc
    flux_vw_sum += flux_vw_blk.mean(dim=dim_t).values * n_bloc
    n_processed += n_bloc
    print(f'  Traité : t={t0}–{t1-1}  ({n_processed} pas de temps au total)')

# Normalisation : moyenne temporelle finale
flux_uw = flux_uw_sum / n_processed  # ρ₀ <u'w'>  [kg m⁻¹ s⁻²]
flux_vw = flux_vw_sum / n_processed

print('\nFlux de Reynolds globaux calculés.')


In [ ]:
# ======================================================================
# 6.2  Tendance (divergence verticale) : -1/ρ₀ · ∂(ρ₀ <u'w'>) / ∂z
# ======================================================================
dz = np.diff(alt)  # épaisseurs des couches [m]
# Gradient centré sur les niveaux intérieurs, différences décentrées aux bords
dflux_uw_dz = np.gradient(flux_uw, alt)  # ∂(ρ₀ <u'w'>) / ∂z
dflux_vw_dz = np.gradient(flux_vw, alt)

rho0_np = rho0.values

# Tendance du vent moyen [m/s²] — terme de Reynolds
tendance_u_global = - dflux_uw_dz / rho0_np
tendance_v_global = - dflux_vw_dz / rho0_np

# ======================================================================
# 6.3  Tracé flux + tendance
# ======================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

ax1 = axes[0]
ax1.plot(flux_uw, alt, color='royalblue', lw=2, label=r"$\rho_0 \overline{u'w'}$")
ax1.plot(flux_vw, alt, color='seagreen',  lw=2, label=r"$\rho_0 \overline{v'w'}$", linestyle='--')
ax1.axvline(0, color='grey', alpha=0.5)
ax1.set_xlabel('Flux de Reynolds ρ₀⟨φ′w′⟩  (kg m⁻¹ s⁻²)')
ax1.set_ylabel('Altitude (m)')
ax1.set_title('A. Flux vertical de quantité de mouvement', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(tendance_u_global * 1e5, alt, color='royalblue', lw=2, label=r"Tendance $u$")
ax2.plot(tendance_v_global * 1e5, alt, color='seagreen',  lw=2, label=r"Tendance $v$", linestyle='--')
ax2.axvline(0, color='grey', alpha=0.5)
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.set_title('B. Tendance du vent par la turbulence\n'
              r'$-\frac{1}{\rho_0}\frac{\partial}{\partial z}(\rho_0\overline{\phi\'w\'})$',
              fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan global de quantité de mouvement — état stationnaire', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Bilan conditionnel — séparation par régime turbulent

On décompose le flux total ρ₀⟨u′w′⟩ en trois contributions physiques :
- **Convection nuageuse** : w′ > 0 et q_c = clw + cli > seuil
- **Thermiques secs** (couche limite) : w′ > 0 et q_c ≤ seuil
- **Subsidence** : w′ < 0
- **Reste** (petite turbulence isotrope) : flux total − somme des trois précédents


In [ ]:
# ======================================================================
# 7.1  Seuils physiques
# ======================================================================
SEUIL_W    = 0.05   # m/s  — updraft significatif
SEUIL_QC   = 1e-5   # kg/kg — présence de condensats

# ======================================================================
# 7.2  Calcul par blocs temporels (même stratégie qu au §6)
# ======================================================================
flux_conv_sum  = np.zeros(len(alt))
flux_sec_sum   = np.zeros(len(alt))
flux_sub_sum   = np.zeros(len(alt))
flux_tot_sum2  = np.zeros(len(alt))
n_proc2 = 0

for t0 in range(t_start, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk   = u.isel({dim_t: sl}).load()
    w_blk   = w.isel({dim_t: sl}).load()
    clw_blk = clw.isel({dim_t: sl}).load()
    cli_blk = cli.isel({dim_t: sl}).load()

    u_prime = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    w_prime = w_blk - w_blk.mean(dim=[dim_y, dim_x])
    qc_blk  = clw_blk + cli_blk

    # Flux local ρ₀ u′w′  (shape : bloc, z, y, x)
    flux_local = rho0 * u_prime * w_prime

    # Masques physiques
    m_conv = (w_prime >  SEUIL_W) & (qc_blk >  SEUIL_QC)
    m_sec  = (w_prime >  SEUIL_W) & (qc_blk <= SEUIL_QC)
    m_sub  = (w_prime < -SEUIL_W)

    # Flux conditionnel = moyenne spatiale du flux là où le masque est vrai
    # On remplace les valeurs hors masque par 0 (contribution nulle)
    flux_conv_blk = flux_local.where(m_conv, 0.0).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_sec_blk  = flux_local.where(m_sec,  0.0).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_sub_blk  = flux_local.where(m_sub,  0.0).mean(dim=[dim_t, dim_y, dim_x]).values
    flux_tot_blk  = flux_local.mean(dim=[dim_t, dim_y, dim_x]).values

    n_bloc = t1 - t0
    flux_conv_sum += flux_conv_blk * n_bloc
    flux_sec_sum  += flux_sec_blk  * n_bloc
    flux_sub_sum  += flux_sub_blk  * n_bloc
    flux_tot_sum2 += flux_tot_blk  * n_bloc
    n_proc2       += n_bloc
    print(f'  Bloc t={t0}–{t1-1}')

flux_conv = flux_conv_sum / n_proc2
flux_sec  = flux_sec_sum  / n_proc2
flux_sub  = flux_sub_sum  / n_proc2
flux_tot  = flux_tot_sum2 / n_proc2
flux_rest = flux_tot - (flux_conv + flux_sec + flux_sub)

print('Flux conditionnels calculés.')


In [ ]:
# ======================================================================
# 7.3  Tendances conditionnelles
# ======================================================================
def tendance(flux_profil):
    """Calcule -1/ρ₀ · d(flux)/dz avec gradient centré."""
    return -np.gradient(flux_profil, alt) / rho0_np

tend_tot  = tendance(flux_tot)
tend_conv = tendance(flux_conv)
tend_sec  = tendance(flux_sec)
tend_sub  = tendance(flux_sub)
tend_rest = tendance(flux_rest)

# ======================================================================
# 7.4  Visualisation
# ======================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 8), sharey=True)

kw_tot  = dict(color='black',   lw=2.5)
kw_conv = dict(color='crimson', lw=1.8, linestyle='--')
kw_sec  = dict(color='orange',  lw=1.8, linestyle='--')
kw_sub  = dict(color='royalblue', lw=1.8, linestyle=':')
kw_rest = dict(color='grey',    lw=1.5, alpha=0.7)

ax1.plot(flux_tot,  alt, label=r"Total $\rho_0 \overline{u'w'}$", **kw_tot)
ax1.plot(flux_conv, alt, label=r"Convection nuageuse ($w'>0,\,q_c>0$)", **kw_conv)
ax1.plot(flux_sec,  alt, label=r"Thermiques secs ($w'>0,\,q_c=0$)",     **kw_sec)
ax1.plot(flux_sub,  alt, label=r"Subsidence ($w'<0$)",                   **kw_sub)
ax1.plot(flux_rest, alt, label='Reste (petite turbulence)',               **kw_rest)
ax1.axvline(0, color='grey', alpha=0.4)
ax1.set_title('A. Flux ρ₀⟨u′w′⟩ par régime turbulent', fontweight='bold')
ax1.set_xlabel('Flux (kg m⁻¹ s⁻²)')
ax1.set_ylabel('Altitude (m)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

scale = 1e5  # mise à l échelle pour la lisibilité
ax2.plot(tend_tot  * scale, alt, label='Total',                **kw_tot)
ax2.plot(tend_conv * scale, alt, label='Convection nuageuse',   **kw_conv)
ax2.plot(tend_sec  * scale, alt, label='Thermiques secs',       **kw_sec)
ax2.plot(tend_sub  * scale, alt, label='Subsidence',            **kw_sub)
ax2.plot(tend_rest * scale, alt, label='Reste',                 **kw_rest)
ax2.axvline(0, color='grey', alpha=0.4)
ax2.set_title('B. Tendance du vent zonal par régime\n'
              r'$-\frac{1}{\rho_0}\frac{\partial}{\partial z}(\rho_0\overline{u\'w\'})$  (×10⁻⁵ m/s²)',
              fontweight='bold')
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan conditionnel — décomposition par régime turbulent\nÉtat stationnaire', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Bilan de quantité de mouvement par région (sec / humide)

On applique le masque sec/humide (défini au §4) pour calculer  
les flux et tendances séparément dans chaque région.


In [ ]:
# ======================================================================
# 8.1  Calcul des flux ρ₀ <u'w'> moyennés sur les colonnes sèches / humides
# ======================================================================
# masque_humide est un DataArray 2D (y, x). On le charge une fois.
mh = masque_humide.values  # booléen (y, x)
ms = masque_sec.values

flux_h_sum = np.zeros(len(alt))  # humide
flux_s_sum = np.zeros(len(alt))  # sec
n_proc3 = 0

for t0 in range(t_start, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk = u.isel({dim_t: sl}).load()
    w_blk = w.isel({dim_t: sl}).load()

    u_prime = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    w_prime = w_blk - w_blk.mean(dim=[dim_y, dim_x])
    flux_local = (rho0 * u_prime * w_prime)  # (bloc, z, y, x)

    # Sélection spatiale selon le masque, puis moyenne
    # mh a la forme (y, x) : on l applique sur les axes y, x du flux
    flux_h_blk = flux_local.values[..., mh].mean(axis=(0, -1))  # (z,)
    flux_s_blk = flux_local.values[..., ms].mean(axis=(0, -1))

    n_bloc = t1 - t0
    flux_h_sum += flux_h_blk * n_bloc
    flux_s_sum += flux_s_blk * n_bloc
    n_proc3    += n_bloc

flux_humide = flux_h_sum / n_proc3
flux_sec_r  = flux_s_sum / n_proc3

tend_humide = tendance(flux_humide)
tend_sec_r  = tendance(flux_sec_r)

print('Flux par région calculés.')


In [ ]:
# ======================================================================
# 8.2  Visualisation
# ======================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)

ax1.plot(flux_tot,    alt, 'k-',  lw=2.5, label='Moyen global')
ax1.plot(flux_humide, alt, color='royalblue', lw=2, linestyle='--', label='Région humide')
ax1.plot(flux_sec_r,  alt, color='saddlebrown', lw=2, linestyle=':', label='Région sèche')
ax1.axvline(0, color='grey', alpha=0.4)
ax1.set_title('Flux ρ₀⟨u′w′⟩ — global vs régions', fontweight='bold')
ax1.set_xlabel('Flux (kg m⁻¹ s⁻²)')
ax1.set_ylabel('Altitude (m)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(tend_tot    * 1e5, alt, 'k-',  lw=2.5, label='Moyen global')
ax2.plot(tend_humide * 1e5, alt, color='royalblue', lw=2, linestyle='--', label='Région humide')
ax2.plot(tend_sec_r  * 1e5, alt, color='saddlebrown', lw=2, linestyle=':', label='Région sèche')
ax2.axvline(0, color='grey', alpha=0.4)
ax2.set_title('Tendance du vent zonal — global vs régions\n(×10⁻⁵ m/s²)', fontweight='bold')
ax2.set_xlabel('Tendance (×10⁻⁵ m/s²)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Bilan de QdM : régions sèches vs humides', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 9. Évolution temporelle du flux de Reynolds intégré verticalement

On suit l'évolution au cours du temps du flux ρ₀⟨u′w′⟩  
intégré sur la troposphère (0–15 km) — indicateur de la maturité de l'agrégation.


In [ ]:
# ======================================================================
# 9.1  Flux intégré verticalement pour chaque pas de temps
# ======================================================================
# On cherche l indice maximal de la troposphère
idx_tropo = np.searchsorted(alt, 15000)  # ~15 km

flux_int_time = []

for t0 in range(0, n_times, BLOC):
    t1 = min(t0 + BLOC, n_times)
    sl = slice(t0, t1)

    u_blk = u.isel({dim_t: sl}).load()
    w_blk = w.isel({dim_t: sl}).load()

    u_prime = u_blk - u_blk.mean(dim=[dim_y, dim_x])
    w_prime = w_blk - w_blk.mean(dim=[dim_y, dim_x])

    # Flux ρ₀ u′w′, moyenné horizontalement, intégré verticalement
    flux_prof = (rho0 * u_prime * w_prime).mean(dim=[dim_y, dim_x])  # (bloc, z)
    flux_int  = flux_prof.isel({dim_z: slice(0, idx_tropo)}).integrate(coord=dim_z)  # (bloc,)
    flux_int_time.append(flux_int.values)

flux_int_time = np.concatenate(flux_int_time)  # (n_times,)

# Axe temporel en jours
time_days = u[dim_t].values.astype('float64') / (1e9 * 3600 * 24)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(time_days, flux_int_time, color='black', lw=1.2)
ax.axvline(time_days[t_start], color='red', lw=1, linestyle='--', label='Début état stationnaire')
ax.set_xlabel('Temps (jours)')
ax.set_ylabel('∫ ρ₀⟨u′w′⟩ dz  (kg/s²)')
ax.set_title('Évolution temporelle du flux de Reynolds intégré verticallement (0–15 km)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Notes méthodologiques

### Gestion mémoire
Tous les fichiers 3D sont ouverts en mode **lazy** (`xr.open_dataset(..., chunks={'time': 10})`).  
Les calculs se font par **blocs de 10 pas de temps** : à chaque itération, seul le bloc courant  
est chargé en RAM, traité, puis libéré. Cette approche rend le code indépendant de la taille  
totale des fichiers.

### Densité de référence ρ₀
Calculée à partir de la loi des gaz parfaits appliquée à l'air humide :
$$\rho_0 = \frac{p}{R_d \, T_v}, \quad T_v = T \cdot \frac{1 + q_v/\varepsilon}{1 + q_v}$$
avec ε = Rd/Rv ≈ 0.622. ρ₀ est ensuite moyenné horizontalement et temporellement  
(sur l'état stationnaire) pour obtenir un profil vertical de référence.

### Tenseur de Reynolds
Le flux est exprimé sous forme **pondérée par la densité** ρ₀⟨u′w′⟩ [kg m⁻¹ s⁻²]  
plutôt que le simple ⟨u′w′⟩ [m² s⁻²]. Cela est nécessaire pour calculer correctement  
la tendance dans l'équation de QdM en coordonnées pression / altitude.

La tendance est :
$$\left(\frac{\partial u}{\partial t}\right)_{\text{Reynolds}} = -\frac{1}{\rho_0} \frac{\partial (\rho_0 \overline{u'w'})}{\partial z}$$

### Classification des régimes turbulents
| Régime | Condition |
|--------|----------|
| Convection nuageuse | w′ > 0.05 m/s  **et**  qc > 10⁻⁵ kg/kg |
| Thermiques secs     | w′ > 0.05 m/s  **et**  qc ≤ 10⁻⁵ kg/kg |
| Subsidence          | w′ < −0.05 m/s |
| Petite turbulence   | résidu |

Ces seuils sont conservatifs et peuvent être ajustés dans les cellules §7 et §8.
